# EuVSOP use case using pyBIS

This Python script aims to implement the EuVSOP use case in BAM Data Store ([main](https://main.datastore.bam.de/), [training](https://training.datastore.bam.de/), [playground](https://playground.datastore.bam.de/), [demo](https://demo.datastore.bam.de/)) instances and meant to be used by **DSSt(s)** during the onboarding process.

More detailed description of the script and used functions can be found in the [Wiki](https://datastore.bam.de/en/pyBIS). More information about the EuVSOP use case and corresponding publication can be found [here](https://datastore.bam.de/en/use_cases/EuVSOP).

### Contents of the script

**Introduction**

- Importing libraries
- Login and personalization
- Setting paths to the folders and Excel files

**Registering Entities** 

Projects, Collections, Objects, and Datasets, including parent-child relationships, and uploading datasets in:
1. Private **division Inventory**
    - Register Projects (Consumables Euvsop, Materials Euvsop, Instruments Euvsop) 
    - Register Collections (Chemicals, Samples, Instruments)
    - Register Objects (Object types: CHEMICAL, SAMPLE, INSTRUMENT)
2. Public **BAM Inventory**
    - Register Collection (Instruments EuVSOP)
    - Register Objects (Object type: INSTRUMENT)
3. **Lab Notebook**
    - Register Project (Eu Vsop)
    - Register Collection of Type Default Experiment (Experimental Steps)
    - Register Objects (Object type: EXPERIMENTAL_STEP) with parent-child relationship(s)
    - Upload datasets (Dataset types: ELN_PREVIEW, OTHER_DATA, ATTACHMENT)

<span style="color: #59a752;"># Comments and important notes are commented out and will be visible below.</span>


## Introduction

### Importing libraries

In [ ]:
from __future__ import annotations

from getpass import getpass
from pathlib import Path
import re

import pandas as pd
from ipywidgets import Dropdown
from pybis import Openbis

### Login and personalization

When running this script for the first time, create a new **personal access token (PAT)**.

PAT will be saved on the disk, i.e. in the ~/.pybis directory. By default this token will be valid for one year.

**Keep you access tokens safe and don't share it with others!** 

Token will be invalidated when:
- Explicit logout() call.
- Number of sessions per user has reached beyond configured limit.
- Session timeout is reached.
- Openbis instance is restarted.

#### Personalization

Enter your BAM **username**

In [ ]:
# Enter your username, e.g. 'MMUSTER' or 'mmuster'
username = ''

#### Login

Select one of the BAM Data Store **instances** in the drop-down menu which would appear after executing the cell below.

In [ ]:
instances = ["main", "training", "playground", "demo"]

dd = Dropdown(options=instances, description="Select an instance:")
display(dd)

In [ ]:
instance = dd.value
o = Openbis(f"https://{instance}.datastore.bam.de")
print(f"Choosen instance (URL): {o.url}")

Execute the cell and enter the **password** in the field that would appear. For subsequent runs, this cell should be **commented out**.

In [ ]:
password = getpass('Enter PASSWORD: ')
o.login(username, password)

#### Generate and save the **PAT**

First, session name needs to be explicitly defined to create corresponding PAT next. We advise to set meaningful name as combination of the project's and instance's name, so it could be easy found and identified later. For subsequent runs, this cell should be **commented out**.

In [ ]:
SESSION_NAME = fr"EuVSOP_{instance}"
pat = o.get_or_create_personal_access_token(sessionName=SESSION_NAME)
o.set_token(pat.permId, save_token=True)

#### Definition of user Space, i.e. your **Lab Notebook**

In [ ]:
division = o.get_object(sample_ident= f'/BAM_GLOBAL/BAM_DATA/PERSONS/{username.upper()}').props('bam_oe').removeprefix('OE_')
my_space = o.get_space(code=fr"{division}_{username}")
print(f'Space of your Lab Notebook: {my_space.code}')

### Setting paths to the folders and Excel files

In the ZIP file or GitHub repository, you will find two folders, one of which contains Excel files. Each Excel file contains metadata of objects and can be used to [batch register](https://datastore.bam.de/en/How_to_guides/Batch_registration_Inventory) them as described in How-to guide. Here, we will extract the objects’ metadata and load it into pandas DataFrames so it can be handled in Python.

To begin with, the paths to each folder and file need to be defined explicitly. The path to the folder containing the datasets is also defined here.

In [ ]:
# Paths to this script
SCRIPT_DIR = Path.cwd()

#Paths to the folder containing the Excel files, and to the Excel files themselves.
excel_folder_path = (SCRIPT_DIR/"20260309_EuVSOP_excel_files")
excel_chemicals = (excel_folder_path/"20260203_EuVSOP_chemicals.xlsx")
excel_samples = (excel_folder_path/"20260203_EuVSOP_samples.xlsx")
excel_instruments = (excel_folder_path/"20260203_EuVSOP_instruments.xlsx")
excel_experimental_steps = (excel_folder_path/"20260305_EuVSOP_experimental_steps.xlsx")

# Paths to datasets folder
datasets_folder_path = (SCRIPT_DIR/"20260203_EuVSOP_datasets")

#### Defining Entity types

All necessary Entity types are defined below, so they can be easily referred later.

In [ ]:
# Collections
type_default_experiment = 'DEFAULT_EXPERIMENT'
type_collection = 'COLLECTION'

# Objects
type_chemical = 'CHEMICAL'
type_sample = 'SAMPLE'
type_instrument = 'INSTRUMENT'
type_experimental_step = 'EXPERIMENTAL_STEP'

# Datasets
type_dataset_eln_preview = 'ELN_PREVIEW'
type_dataset_other_data = 'OTHER_DATA'
type_dataset_attachment = 'ATTACHMENT'

#### Data from Excel files transferred into DataFrames

Data from each Excel spreadsheet is transferred into a separate DataFrame.

In [ ]:
chemicals_df = pd.read_excel(excel_chemicals, sheet_name=type_chemical)
samples_df = pd.read_excel(excel_samples, sheet_name=type_sample)
instruments_df = pd.read_excel(excel_instruments, sheet_name=type_instrument)
experimental_steps_df = pd.read_excel(excel_experimental_steps, sheet_name=type_experimental_step)

#### Function to convert a DataFrame to a list of dictionaries

In this cell data will be converted from DataFrame to a list of dictionaries. After executing this cell you should see the number of objects with the corresponding metadata in each list and make sure they are correspond to number of objects in Excel.

In [ ]:
def df_to_list_of_dicts(df: pd.DataFrame) -> list[dict]:
    def _clean(v):
        if pd.isna(v):
            return None
        if isinstance(v, (pd.Timestamp,)):
            return v.isoformat()
        if isinstance(v, str):
            s = v.strip()
            return s if s else None
        return v

    # find header row (row that contains a "Name" column label, case-insensitive)
    header_idx = None
    for i, row in df.iterrows():
        if row.astype(str).str.strip().str.casefold().eq("name").any():
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not find header row containing 'Name'")

    # set headers from the header row and keep rows below it
    headers = df.iloc[header_idx].astype(str).str.strip().tolist()
    df2 = df.iloc[header_idx + 1 :].reset_index(drop=True).copy()
    df2.columns = headers
    df2 = df2.dropna(how="all")

    # if a "$" column exists and contains $1, $2... keep only those rows, then drop the "$" column
    if "$" in df2.columns:
        s = df2["$"].astype(str).str.strip()
        if s.str.match(r"^\$\d+$", na=False).any():
            df2 = df2[s.str.match(r"^\$\d+$", na=False)].reset_index(drop=True)
        df2 = df2.drop(columns=["$"], errors="ignore")

    # convert DataFrame to list of dicts, skipping NaN/empty values
    result = []
    for _, r in df2.iterrows():
        entry = {}
        for col in df2.columns:
            val = _clean(r.get(col))
            if val is None:
                continue

            key = str(col).strip()

            if key.casefold() == "name":
                key = "$name"
            elif key.casefold() == "Alternative name".casefold():
                key = "alias"
            elif key.casefold() == "IUPAC Name".casefold():
                key = "iupac_name"
            elif key.casefold() == "CAS Registry Number".casefold():
                key = "cas_number"
            elif key.casefold() == "Manufacturer".casefold(): 
                key = "manufacturer"
            elif key.casefold() == "Description".casefold():
                key = "description"
            elif key.casefold() == "Hazardous Substance".casefold():
                key = "hazardous_substance"
            elif key.casefold() == "BAM Organizational Entity".casefold():
                key = "bam_oe"
            elif key.casefold() == "Complete BAM Location".casefold():
                key = "bam_location_complete"

            elif key.casefold() == "Notes".casefold():
                key = "notes"
            elif key.casefold() == "Show in project overview".casefold():
                key = "$show_in_project_overview"
                val = bool(val) if isinstance(val, str) else bool(val)
            elif key.casefold() == "Experiment completed".casefold():
                key = "finished_flag"
                val = bool(val) if isinstance(val, str) else bool(val)
            elif key.casefold() == "Start date".casefold():
                key = "start_date"
            elif key.casefold() == "End date".casefold():
                key = "end_date"

            elif key.casefold() == "Experimental goals".casefold():
                key = "experimental_step.experimental_goals"
            elif key.casefold() == "Experimental description".casefold():
                key = "experimental_step.experimental_description"               
            elif key.casefold() == "Experimental results".casefold():
                key = "experimental_step.experimental_results"
            elif key.casefold() == "Spreadsheet".casefold():
                key = "experimental_step.spreadsheet"

            elif key.casefold() == "References".casefold():
                key = "reference"     
            elif key.casefold() == "Publication".casefold():
                key = "publication"     


            entry[key] = val

        # ensure canonical $name key exists
        name_val = entry.get("$name") or entry.get("Name") or entry.get("name")
        if name_val:
            entry["$name"] = name_val

        result.append(entry)

    return result


chemicals_data = df_to_list_of_dicts(chemicals_df)
print("Number of objects in chemicals_data:", len(chemicals_data))
print("First chemical:", chemicals_data[:1])

samples_data = df_to_list_of_dicts(samples_df)
print("Number of objects in samples_data:", len(samples_data))
print("First sample:", samples_data[:1])

instruments_data = df_to_list_of_dicts(instruments_df)
print("Number of objects in instruments_data:", len(instruments_data))
print("First instrument:", instruments_data[:1])

experimental_steps_data = df_to_list_of_dicts(experimental_steps_df)
print("Number of objects in experimental_steps_data:", len(experimental_steps_data))
print("First Experimental Step:", experimental_steps_data[:1])


## Registering Entities

Projects, Collections, Objects, and Datasets, including parent-child relationships, and uploading datasets in:

### 1. Private division Inventory

#### Register Projects

**Only DSSt(s) can create new Projects in division Inventory Spaces**

Three projects named **Consumables Euvsop**, **Materials Euvsop**, and **Instruments Euvsop** will be registered by executing the cell below if they do not already exist in the corresponding Spaces. Verification is performed using the Project `identifier`, which is formed from the Space code and the Project code.

In [ ]:
def get_or_create_project(space: str, code: str):
    project_id = f"/{space}/{code}"
    try:
        project = o.get_project(projectId=project_id)
        print(f"Already exists: {project.identifier}")
        return project
    except ValueError:
        project = o.new_project(space=space, code=code)
        project.save()
        print(f"Registered: {project.identifier}")
        return project

project_instruments = get_or_create_project(f"{division}_EQUIPMENT", "INSTRUMENTS_EUVSOP")
project_material    = get_or_create_project(f"{division}_MATERIALS",  "MATERIALS_EUVSOP")
project_consumable  = get_or_create_project(f"{division}_MATERIALS",  "CONSUMABLES_EUVSOP")

#### Register Collections

Three Collection (with names **Chemicals**, **Samples**, and **Instruments**) will be registered in the corresponding Projects unless they are already exists. 
Verification is also performed using the Collection `identifier`, which consists of the Project `identifier` and the Collection code.

**Note**: In openBIS, an Experiment corresponds to a Collection — both terms refer to the same concept. Once a Collection has been successfully registered, it is automatically marked as a successfully registered Experiment in the system.

In [ ]:
def get_or_create_collection(code: str, name: str, default_type, project_id: str):
    collection_id = f"{project_id}/{code}"
    try:
        collection = o.get_collection(code=collection_id)
        print(f"Collection already exists: Name: {collection.props.get('$name')} | {collection.identifier}")
        return collection
    except ValueError:
        collection = o.new_collection(
            code=code,
            type=type_collection,
            project=project_id,
            props={"$name": name, "$default_object_type": default_type},
        )
        collection.save()
        print(f"Registered: {collection.identifier}")
        return collection

collection_chemicals   = get_or_create_collection("COLLECTION_CHEMICALS",   "Chemicals",   type_chemical,   project_consumable.identifier)
collection_samples     = get_or_create_collection("COLLECTION_SAMPLES",     "Samples",     type_sample,     project_material.identifier)
collection_instruments = get_or_create_collection("COLLECTION_INSTRUMENTS", "Instruments", type_instrument, project_instruments.identifier)

#### Register Objects

Object type: CHEMICAL

First, the Object name is compared with the names of all Objects in the Collection to ensure there are no duplicates. If the name passes this check, a new Object with that name is created using the corresponding metadata from the dictionary.

In [ ]:
# Names normalization function, so all names are compared in a uniform way
def _norm_name(v: str) -> str:
    return " ".join((v or "").split()).strip().casefold()

# Build index of existing chemicals by normalized name
def build_chemical_name_index():
    index = {}
    for obj in o.get_objects(type=type_chemical, collection=collection_chemicals.identifier):
        key = _norm_name(obj.props.get("$name"))
        if key:
            index[key] = obj
    return index

# Chemical registration function if not exists already. Verifies by normalized name.
def get_or_create_chemical_by_name(chem: dict, index: dict):
    name = (chem.get("$name") or chem.get("Name") or chem.get("name") or "").strip()
    key = _norm_name(name)
    if not key:
        raise ValueError("Chemical name missing (expected chem['$name'] or chem['Name'] or chem['name']).")

    existing = index.get(key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(chem)
 
    obj = o.new_object(
        type=type_chemical,
        collection=collection_chemicals.identifier,
        props=props,
    )
    obj.save()
    index[key] = obj
    print(f"Registered: {obj.identifier} | {name}")
    return obj

chemical_index = build_chemical_name_index()
chemicals = [get_or_create_chemical_by_name(chem, chemical_index) for chem in chemicals_data]


- Register multiple Objects with one parent

Object type: SAMPLE

First, an Object named “EuVSOP” is registered if it does not already exist. The check is performed using the Object name.

In [ ]:
list_of_existing_samples = []
for obj in o.get_objects(collection=collection_samples.identifier, type=type_sample):
    list_of_existing_samples.append(obj.props.get("$name"))

if "EuVSOP" in list_of_existing_samples:
    sample_euvsop_list = o.get_objects(collection=collection_samples.identifier, type=type_sample)

    sample_euvsop = None
    for obj in sample_euvsop_list:
        if (obj.props.get("$name") or "").strip() == "EuVSOP":
            sample_euvsop = obj
            break

    if sample_euvsop is None:
        raise ValueError("Sample 'EuVSOP' was listed but could not be resolved to an object.")

    print(f"Sample already exists: EuVSOP | {sample_euvsop.identifier}")

else:
    props = next((s for s in samples_data if s.get("$name") == "EuVSOP"), None)
    if props is None:
        raise ValueError("No sample '$name' == 'EuVSOP' found in samples_data")

    sample_euvsop = o.new_object(
        type=type_sample,
        collection=collection_samples.identifier,
        props=props,
    )
    sample_euvsop.save()
    print(f"Created sample: {sample_euvsop.identifier} | {sample_euvsop.props.get('$name')}")

Next, all sub-samples with the parent sample EuVSOP are registered.

In [ ]:
for samp in samples_data:
    parents_val = samp.get("Parents") or samp.get("parents")
    if parents_val is None:
        print(f"Skipping sample without 'Parents': {samp.get('$name')}")
        continue

    if str(parents_val).strip() != "$1":
        continue

    samp_name_key =samp.get("$name")

    if samp_name_key in list_of_existing_samples:
        print(f"Sample already exists: {samp.get('$name')}")
        continue

    props = dict(samp)
    props.pop("Parents", None)
    props.pop("parents", None)

    obj = o.new_object(
        type=type_sample,
        collection=collection_samples.identifier,
        parents=[sample_euvsop.identifier],
        props=props,
    )
    obj.save()
    list_of_existing_samples.append(samp_name_key)
    print(f"Created sample: {obj.identifier} | {obj.props.get('$name')}")

One sub-sample will be selected for the Experimental Step: EuVSOP HEE Treatment.

In [ ]:
selected_sample_name = None

for name in list_of_existing_samples:
    if name and name.startswith("EuVSOP_"):
        if selected_sample_name is None:
            selected_sample_name = name
        else:
            print("More than one sample starts with 'EuVSOP_'. Using the first match.")
            break

if selected_sample_name is None:
    print("No 'EuVSOP_' sample found.")
else:
    # make comparison robust (strip spaces + non-breaking spaces)
    target = str(selected_sample_name).replace("\u00A0", " ").strip()

    selected_sample = None
    for obj in o.get_objects(collection=collection_samples.identifier, type=type_sample):
        obj_name = str(obj.props.get("$name") or "").replace("\u00A0", " ").strip()
        if obj_name == target:
            selected_sample = obj
            print(f"Selected sample: {selected_sample.identifier} | {selected_sample.props.get('$name')}")
            break

    if selected_sample is None:
        raise ValueError(f"Could not resolve selected_sample_name to an object: {target!r}")

- Register Objects

Object type: INSTRUMENT

Each Object will be registered if it does not already exist. The check is performed using the Object name.

In [ ]:
# Build index of existing instruments by normalized name
def build_instrument_name_index():
    index = {}
    for obj in o.get_objects(type=type_instrument, collection=collection_instruments.identifier):
        key = _norm_name(obj.props.get("$name"))
        if key:
            index[key] = obj
    return index

# Instrument registration function if not exists already. Verifies by normalized name.
def get_or_create_instrument_by_name(ins: dict, index: dict):
    name = (ins.get("$name") or ins.get("Name") or ins.get("name") or "").strip()
    key = _norm_name(name)
    if not key:
        raise ValueError("Instrument name missing (expected ins['$name'] or ins['Name'] or ins['name']).")

    existing = index.get(key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(ins)

    obj = o.new_object(
        type=type_instrument,
        collection=collection_instruments.identifier,
        props=props,
    )
    obj.save()
    index[key] = obj
    print(f"Registered: {obj.identifier} | {name}")
    return obj

instrument_index = build_instrument_name_index()
instruments = [get_or_create_instrument_by_name(ins, instrument_index) for ins in instruments_data]

### 2. Public BAM Inventory

#### Register Collection

In the public BAM Inventory Project, a Collection with the name Instruments EuVSOP will be registered unless it already exists. Verification is done by `identifier` of the Collection.

In [ ]:
def get_or_create_collection(code: str, name: str, default_type, project_id: str):
    collection_id = f"{project_id}/{code}"
    try:
        collection = o.get_collection(code=collection_id)
    except Exception:
        collection = None

    if collection is None:
        collection = o.new_collection(
            code=code,
            type=type_collection,
            project=project_id,
            props={"$name": name, "$default_object_type": default_type},
        )
        collection.save()
        print(f"Registered: {collection.identifier}")
    else:
        print(f"Already exists: Name: {collection.props["$name"]} with identifier: {collection.identifier}")

    return collection

collection_bam_inst = get_or_create_collection("COLLECTION_INSTRUMENTS_EUVSOP", "Instruments EuVSOP",type_instrument,f"/BAM_EQUIPMENT/{division}_EQUIPMENT_OPEN")

#### Register Objects

Object type: INSTRUMENT

First, Properties of each instrument should be defined explicitly.

In [ ]:
instruments_bam_data = [
    {
        '$name': 'Spectrophotometer',
        'manufacturer': 'SPECORD 205',
        'bam_oe': 'OE_VP.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    },
    {
        '$name': 'Magnetic resonance spectrometer (Minispec mq 40)',
        'manufacturer': 'Bruker',
        'bam_oe': 'OE_VP.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    }
]

Next, Objects will be registered unless they are already existing in the Collection. Verification is done by name.

In [ ]:
def build_instrument_index():
    index = {}
    for obj in o.get_objects(collection=collection_bam_inst.identifier, type=type_instrument):
        name = _norm_name(obj.props.get("$name"))
        if name:
            index[name] = obj
    return index

def get_or_create_instrument(ins: dict, index: dict):
    name_raw = ins.get("$name") or ins.get("name")
    name_key = _norm_name(name_raw)
    if not name_key:
        raise ValueError("Instrument name missing (expected ins['$name'] or ins['name']).")

    existing = index.get(name_key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(ins)
    props["$name"] = (name_raw or "").strip()

    obj = o.new_object(
        type=type_instrument,
        collection=collection_bam_inst.identifier,
        props=props,  # autogenerated
    ) 
    obj.save()
    index[name_key] = obj
    print(f"Registered: {obj.identifier} | {props['$name']}")
    return obj

instrument_index = build_instrument_index()
instruments = [get_or_create_instrument(ins, instrument_index) for ins in instruments_bam_data]

### 3. Lab Notebook

- List all registered Projects in your Lab Notebook Space

In [ ]:
o.get_projects( space = my_space.code)

#### Register Project

The project with the `code` "Eu_VSOP" will be registered in your Lab Notebook Space unless it is already registered. Verification is done by the `identifier` of the Project.

In [ ]:
def get_or_create_project_by_code(space: str, code: str, description: str):
    project_id = f"/{space}/{code}"
    try:
        project = o.get_project(projectId=project_id)
        print(f"Already exists: {project.identifier}")
        return project
    except ValueError:
        project = o.new_project(space=space, code=code, description=description)
        project.save()
        print(f"Registered: {project.identifier}")
        return project

EuVSOP_project = get_or_create_project_by_code(
    space=my_space.code,
    code="EU_VSOP",
    description="The Eu-VSOP project investigates the unambiguous identification of iron oxide nanoparticles -VSOP doped with Europium(III) for flourescence detection in biological samples such as histological tissue sections. Background: VSOP are very small iron oxide nanoparticles used in magnetic resonance imaging (MRI). These nanoparticles are studied as an alternative to Gadolinium based MRI-contrast agents due to their potentially lower toxicity. The clear detection of EuVSOP in tissue sections enables biodistribution studies. Note that the content of this Demo Project are inspired by some scientific open access publications. Some modifications might be included for illustration of openBIS functions."
)

#### Register Collection

Collection type: DEFAULT_EXPERIMENT

In the Project, Collection with the name Experimental Steps will be registered, unless it already exists. Verification is done by the `identifier` of the Collection.

In [ ]:
def get_or_create_collection(code: str, name: str, project_id: str, description: str):
    collection_id = f"{project_id}/{code}"
    try:
        collection = o.get_collection(code=collection_id)
        print(f"Collection already exists: Name: {collection.props.get('$name')} | {collection.identifier}")
        return collection
    except ValueError:
        collection = o.new_collection(
            code=code,
            type=type_default_experiment,
            project=project_id,
            props={"$name": name, "default_experiment.experimental_description": description},
        )
        collection.save()
        print(f"Registered: {collection.identifier}")
        return collection

EuVSOP_collection = get_or_create_collection("EXPERIMENTAL_STEPS",   "Experimental Steps",  EuVSOP_project.identifier, 'This collection contains all the experimental steps done in the Eu-VSOP project: 1.Synthesis of Eu-VSOP (Europium-Very small iron oxide nanoparticles). 2.Treatment of Eu-VSOP with Enhancer solution for detection of Europium. 3.Characterization of Chemical, Magnetic and Optical (fluorescent) properties of Eu-VSOP treated with Enhancer solution.')

#### Register Objects

Object type: EXPERIMENTAL_STEP

A new object will be registered with the metadata from the dictionary, unless it already exists in the Collection. Verification is done by the name of an Object.

In [ ]:
def build_obj_index():
    index = {}
    for obj in o.get_objects(collection=EuVSOP_collection.identifier, type=type_experimental_step):
        name = _norm_name(obj.props.get("$name"))
        if name:
            index[name] = obj
    return index

def get_or_create_experimental_step(ins: dict, index: dict):
    name_raw = ins.get("$name") or ins.get("name")
    name_key = _norm_name(name_raw)
    if not name_key:
        raise ValueError("Experimental Step name missing (expected ins['$name'] or ins['name']).")

    existing = index.get(name_key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(ins)
    props["$name"] = (name_raw or "").strip()

    obj = o.new_object(
        type=type_experimental_step,
        collection=EuVSOP_collection.identifier,
        props=props,  # autogenerated
    ) 
    obj.save()
    index[name_key] = obj
    print(f"Registered: {obj.identifier} | {props['$name']}")
    return obj

experimental_step_index = build_obj_index()
experimental_steps = [get_or_create_experimental_step(ins, experimental_step_index) for ins in experimental_steps_data]

- Finding `identifier` of an Object by its prefix (case-insensitive)

To establish a **[parent-child relationship](https://datastore.bam.de/e//en/concepts/parent-child_relationship)**, the `identifier` of each parent(s) and/or child(ren) Object should be listed. The following function below will return identifier(s) when prefix(es) are given. It's case-insensitive and removes specific characters in a name search of an Object. This function will be used to find and list parent(s) and/or child(ren) to connect the Objects.

In [ ]:
def clean_text(s: str) -> str:
    s = "" if s is None else str(s)

    # replace specific chars with space
    for ch in ['"', "\\", "'", "`", "“", "”", "‘", "’", "™", "®"]:
        s = s.replace(ch, " ")

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_identifiers_by_prefixes_in_collections(collections, prefixes):
    prefixes_key = [clean_text(p).casefold() for p in prefixes]
    matched_ids = []

    for collection_path in collections:
        for obj in o.get_objects(collection=collection_path):
            name = obj.props.get("$name")
            if not name:
                continue

            name_key = clean_text(name).casefold()
            #print(name_key)
            if any(name_key.startswith(p) for p in prefixes_key):
                matched_ids.append(obj.identifier)

    return matched_ids

#list of collection where object's identifiers should be searched
collections = [ 
    collection_chemicals.identifier,
    collection_samples.identifier,
    collection_instruments.identifier,
    collection_bam_inst.identifier 
]

- Updating Experimental Steps with parent-child relationships

By updating the registered Objects with a parent-child relationship, the Objects will be connected.

In [ ]:
for obj in o.get_objects(type=type_experimental_step, collection=EuVSOP_collection.identifier):
    if obj.props.get("$name") == "EuVSOP Synthesis":
        synthesis = obj
    if obj.props.get("$name") == "EuVSOP HEE Treatment":
        treatment = obj
    if obj.props.get("$name") == "EuVSOP Nanoparticle Iron Quantification":
        iron_quantification = obj
    if obj.props.get("$name") == "EuVSOP Nanoparticle Size":
        nanoparticle_size = obj
    if obj.props.get("$name") == "EuVSOP Nanoparticle Magnetic Characterization":
        magnetic_characterization = obj
    if obj.props.get("$name") == "EuVSOP Fluorescence":
        fluorescence = obj

trans = o.new_transaction()
synthesis.add_parents(find_identifiers_by_prefixes_in_collections(collections,
         ["Iron(III)","Iron(II)", "Ammonia", "Citric", "Sodium", "Yttrium", "Vivaflow", "Europium(III) chloride", "Hydrochloric", "Europium(III) standard", #chemicals
           "ultrafiltration", "pH meter", "conductivity", "photo"])) #instruments
synthesis.add_children([f"{sample_euvsop.identifier}"])
treatment.add_parents([f"{selected_sample.identifier}"] +find_identifiers_by_prefixes_in_collections(collections, ["Enhancer"]))
iron_quantification.add_parents([f"{treatment.identifier}"] + find_identifiers_by_prefixes_in_collections(collections, ["Spectrophotometer"]))
nanoparticle_size.add_parents([f"{treatment.identifier}"] + find_identifiers_by_prefixes_in_collections(collections, ["Zetasizer"]))
magnetic_characterization.add_parents([f"{treatment.identifier}"] + find_identifiers_by_prefixes_in_collections(collections, ["Magnetic resonance spectrometer"]))
fluorescence.add_parents([f"{magnetic_characterization.identifier}"] + find_identifiers_by_prefixes_in_collections(collections, ["Hitachi Fluorescence Spectrometer"]))
trans.add(synthesis)
trans.add(treatment)
trans.add(iron_quantification)
trans.add(nanoparticle_size)
trans.add(magnetic_characterization)
trans.add(fluorescence)    
trans.commit()

#### Upload Datasets

Dataset type: Other data

- Uploading Dataset in Experimental Step: EuVSOP Synthesis

In [ ]:
dataset_synthesis = o.new_dataset(
    type       = type_dataset_other_data,
    collection = EuVSOP_collection,
    sample     = f"{synthesis.identifier}",
    files      = [str(datasets_folder_path / 'Synthesis_Solution_Calculations.xlsx')],
    props      = {'$name': 'Solution Calculations', }
)
dataset_synthesis.save()

Dataset type: ELN Preview 

- Uploading Dataset in Experimental Step: EuVSOP HEE Treatment

In [ ]:
dataset_treatment = o.new_dataset(
    type       = type_dataset_eln_preview,
    collection = EuVSOP_collection,
    sample     = f"{treatment.identifier}",
    files      = [str(datasets_folder_path /'Enhancer_EuVSOP.png')],
    props      = {'$name': 'Enhancer EuVSOP', }
)
dataset_treatment.save()

Dataset type: Attachment

- Uploading Dataset in Experimental Step: EuVSOP Nanoparticle Iron Quantification

In [ ]:
dataset_iron_quantification = o.new_dataset(
    type       = type_dataset_attachment,
    collection = EuVSOP_collection,
    sample     = f"{iron_quantification.identifier}",
    files      = [str(datasets_folder_path /'Iron_Quantification.csv')],
    props      = {'$name': 'EuVSOP Iron quantification', }
)
dataset_iron_quantification.save()

Dataset type: ELN Preview
- Uploading Dataset in Experimental Step: EuVSOP Nanoparticle Iron Quantification

In [ ]:
dataset_iron_quantification = o.new_dataset(
    type       = type_dataset_eln_preview,
    collection = EuVSOP_collection,
    sample     = f"{iron_quantification.identifier}",
    files      = [str(datasets_folder_path /'Iron_quantification_Standard_at_510nm.png')],
    props      = {'$name': 'Iron standard at 510 nm', }
)
dataset_iron_quantification.save()

Dataset type: ELN Preview

- Uploading Datasets in Experimental Step: EuVSOP Nanoparticle Size

In [ ]:
dataset_nanoparticle_size = o.new_dataset(
    type       = type_dataset_eln_preview,
    collection = EuVSOP_collection,
    sample     = f"{nanoparticle_size.identifier}",
    files      = [str(datasets_folder_path /'Size_selected_areas_of_electron_diffraction.png')],
    props      = {'$name': 'Size selected areas of electron diffraction', }
)
dataset_nanoparticle_size.save()

Dataset type: ELN Preview

- Uploading Datasets in Experimental Step: EuVSOP Fluorescence

In [ ]:
dataset_fluorescence = o.new_dataset(
    type       = type_dataset_eln_preview,
    collection = EuVSOP_collection,
    sample     = f"{fluorescence.identifier}",
    files      = [str(datasets_folder_path /'FluorescenceMeasurement.png')],
    props      = {'$name': 'EuVSOP Fluorescence', }
)
dataset_fluorescence.save()